# 2. Cell type annotation

The input from `0_integrate_two_datasets` is `adata1.h5ad` (130k cells integrated with Harmony and clustered with Leiden).

Workflow
1. Load the data and inspect the Leiden clusters.
2. Plot known GBM markers to assign preliminary cluster identities.
3. Use `rank_genes_groups` to verify cluster-specific markers.
4. Annotate the clusters manually and save `adata2.h5ad`.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

sc.set_figure_params(dpi=100, color_map='viridis_r', transparent=False, frameon=False)
%config InlineBackend.figure_format = 'retina'

WRITE_DIR = Path('./write/0_integrate_two_datasets')
FIG_DIR   = Path('./figures/2_marker_gene')
FIG_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(FIG_DIR)

print(f'scanpy {sc.__version__}')

## 1. Load the data

In [ ]:
adata = sc.read(WRITE_DIR / 'adata1.h5ad')

if 'log1p' in adata.uns:
    adata.uns['log1p']['base'] = None

adata

In [ ]:
# Inspect the basic distributions.
print('Leiden clusters:', adata.obs['leiden'].nunique())
print(adata.obs['leiden'].value_counts().sort_index())
print()
print('Tissue:', adata.obs['tissue_type'].value_counts())
print()
print('Dataset:', adata.obs['dataset'].value_counts())

In [ ]:
sc.pl.umap(adata, color=['leiden', 'tissue_type', 'dataset'],
           ncols=3, legend_loc='on data', legend_fontsize=6)

## 2. Visualize known GBM marker genes

In [ ]:
# GBM marker genes, v3: address overassignment of the CDC label.
gbm_markers = {
    # --- Tumor & Glia ---
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN", "NOVA1", "NRCAM", "TUBB2B"],
    "Prol.tumor":     ["SOX2", "SOX6", "MKI67", "TOP2A", "CCDC34", "PBK", "UBE2T", "CENPV", "CENPF"],
    "Oligodendrocyte":["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "Neuronal":       ["SYT1", "STMN2", "RBFOX3", "GRIN1", "SLC17A7", "GAD1", "GAD2"],
    # --- Lymphoid ---
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    # --- Vascular / Stromal ---
    "Pericyte":       ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial":    ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],
    # --- Myeloid --- 
    # v3: strengthen the macrophage panel (add C1Q/APOE/CD163), narrow the CDC panel (remove HLA-DPB1), and add homeostatic microglial markers.
    "Monocyte":       ["FCN1", "VCAN", "S100A8", "S100A9", "CD14"],
    "Microglia":      ["TMEM119", "CX3CR1", "P2RY12", "TREM2", "GPR34", "CTSD", "CSF1R"],
    "Macrophage":     ["CD68", "LYZ", "C1QA", "C1QB", "C1QC", "APOE", "CD163", "MRC1", "GPNMB", "SPP1"],
    "CDC":            ["FCER1A", "CD1C", "CLEC9A", "LAMP3", "IRF8"],
    "Neutrophil":     ["S100A8", "FCGR3B", "IFITM2", "CSF3R", "CXCR2"],
}

# Filter out genes absent from the expression matrix.
markers_filt = {}
for ct, genes in gbm_markers.items():
    found = [g for g in genes if g in adata.var_names]
    markers_filt[ct] = found
    missing = set(genes) - set(found)
    if missing:
        print(f'⚠️ {ct}: missing {missing}')

print(f'\nAvailable marker genes: {sum(len(v) for v in markers_filt.values())}')

In [ ]:
# Dotplot: v2 marker × leiden cluster
sc.pl.dotplot(adata, markers_filt, groupby='leiden',
              standard_scale='var', dendrogram=True,
              save='_gbm_markers_v2_by_leiden.png')

### Coarse annotation

In [ ]:
# Score cells using the specified cell type categories.
gbm_markers_v3 = {
    # --- Tumor & Glia ---
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN", "NOVA1", "NRCAM", "TUBB2B"],
    "Prol.tumor":     ["SOX2", "SOX6", "MKI67", "TOP2A", "CCDC34", "PBK", "UBE2T", "CENPV", "CENPF"],
    "Oligodendrocyte":["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "Neuronal":       ["SYT1", "STMN2", "RBFOX3", "GRIN1", "SLC17A7", "GAD1", "GAD2"],
    # --- Lymphoid ---
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "CD52", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    # --- Vascular / Stromal ---
    "Pericyte":       ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial":    ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],
    # --- Myeloid & Dendritic ---
    "Monocyte":       ["FCN1", "VCAN", "CD52", "S100A8", "S100A9"],
    "Microglia":      ["TMEM119", "CX3CR1", "TREM2", "GPNMB", "CTSD", "CD68"],
    "Macrophage":     ["LYZ", "FN1", "VCAN"],
    "CDC":            ["FCER1A", "HLA-DPB1", "CD83", "AREG"],
    "Neutrophil":     ["S100A8", "FCGR3B", "IFITM2"],
}

# Calculate marker scores.
for ct, genes in gbm_markers_v3.items():
    valid = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(adata, gene_list=valid, score_name=f'score_{ct}')
    missing = set(genes) - set(valid)
    status = f"({len(valid)}/{len(genes)})"
    if missing:
        status += f" missing: {missing}"
    print(f"  {ct}: {status}")

# Average the scores within each cluster.
score_cols = [f'score_{ct}' for ct in gbm_markers_v3]
df_v3 = adata.obs.groupby('leiden', observed=False)[score_cols].mean()
df_v3.columns = list(gbm_markers_v3.keys())

# Display scores and assign the highest-scoring type.
assign_v3 = df_v3.idxmax(axis=1).to_dict()
print("\nAutomatic assignment by highest score:")
for cl in sorted(assign_v3, key=int):
    scores = df_v3.loc[cl]
    top = scores.max()
    second = scores.nlargest(2).iloc[1]
    flag = " ⚠️" if (top - second) < 0.1 else ""
    print(f"  Cluster {cl:>2s} → {assign_v3[cl]:<16s} (top={top:.3f}, 2nd={scores.nlargest(2).index[1]}={second:.3f}){flag}")

print(f"\nScore matrix:")
print(df_v3.round(3).to_string())

In [ ]:
# Inspect the assigned labels.
for cl in sorted(assign_v3, key=int):
    scores = df_v3.loc[cl]
    top = scores.max()
    second = scores.nlargest(2).iloc[1]
    flag = " ⚠️" if (top - second) < 0.1 else ""
    print(f"  Cluster {cl:>2s} → {assign_v3[cl]:<16s} (top={top:.3f}, 2nd={scores.nlargest(2).index[1]}={second:.3f}){flag}")

In [ ]:
# UMAP: compare the revised annotation with Leiden clusters.
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cell_type', ax=axes[0],
           title='Cell type annotation (fixed)', show=False, frameon=False,
           legend_loc='on data', legend_fontsize=8)
sc.pl.umap(adata, color='leiden', ax=axes[1],
           title='Leiden clusters', show=False, frameon=False,
           legend_loc='on data', legend_fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_celltype_annotation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Manual curation based on the v3 dot plot and the top Wilcoxon markers.
manual_override = {
    '0': 'Tumor',
    '1': 'Prol.tumor',    # TOP2A, CENPF, RRM2, ASPM: proliferating tumor cells
    '2': 'Macrophage',    # JUN, FOS, CCL3, CD83: activated TAMs
    '4': 'Macrophage',    # CD83, OLR1, KLF6, CCL3, CD74: activated TAMs
    '5': 'Macrophage',    # C1QB, C1QA, C1QC, HLA-DRA, CD74: complement-expressing TAMs
    '7': 'Macrophage',    # MRC1, MERTK, CSF1R, STAB1, TLR2: M2-type TAMs
    '10': 'Microglia',    # Predominantly HSPs (stress response); near cluster 0 on the UMAP
}

# Apply the revised assignments.
assign_fixed = assign_v3.copy()
assign_fixed.update(manual_override)

adata.obs['cell_type'] = adata.obs['leiden'].map(assign_fixed).astype('category')
print(adata.obs['cell_type'].value_counts())

In [ ]:
from sklearn.neighbors import KernelDensity

def _repel_labels(positions, min_dist=1.5, iterations=50):
    """Move labels apart when their positions are too close."""
    pos = np.array(positions, dtype=float)
    for _ in range(iterations):
        moved = False
        for i in range(len(pos)):
            for j in range(i + 1, len(pos)):
                d = pos[j] - pos[i]
                dist = np.linalg.norm(d)
                if dist < min_dist and dist > 0:
                    shift = (min_dist - dist) / 2 * d / dist
                    pos[i] -= shift
                    pos[j] += shift
                    moved = True
        if not moved:
            break
    return pos

def enhance_umap(adata, color='cell_type', figsize=(8, 8), size=15,
                 kde_bandwidth=0.3, contour_level_frac=0.1,
                 label_fontsize=10, arrow_frac=0.10, palette=None,
                 label_min_dist=2.0, show_pct=True, show_arrows=True,
                 contour_color='gray', contour_style='--',
                 title=None):
    """Combine UMAP styling from Shengxin Dazahui (Bioinformatics Medley) and Shengxin Xingqiu (Bioinformatics Planet)."""
    fig, ax = plt.subplots(figsize=figsize)

    sc.pl.umap(adata, color=color, legend_loc=None, frameon=False,
               size=size, ax=ax, add_outline=True, palette=palette,
               outline_width=(0.1, 0.05), show=False)

    umap_coords = adata.obsm['X_umap']
    categories = adata.obs[color].cat.categories
    colors_list = adata.uns.get(f'{color}_colors', [f'C{i}' for i in range(len(categories))])

    # Cell type proportions
    ct_counts = adata.obs[color].value_counts(normalize=True) * 100

    # Collect label positions.
    label_info = []
    for i, ct in enumerate(categories):
        mask = adata.obs[color] == ct
        coords = umap_coords[mask]
        if len(coords) < 30:
            continue
        mx, my = np.median(coords, axis=0)
        label_info.append((ct, mx, my, i, coords))

    # Move overlapping labels apart.
    raw_positions = [(info[1], info[2]) for info in label_info]
    adjusted = _repel_labels(raw_positions, min_dist=label_min_dist)

    # KDE contours and labels
    for idx, (ct, _, _, ci, coords) in enumerate(label_info):
        kde = KernelDensity(bandwidth=kde_bandwidth, metric='euclidean')
        kde.fit(coords)
        x_min_ct, x_max_ct = coords[:, 0].min() - 0.5, coords[:, 0].max() + 0.5
        y_min_ct, y_max_ct = coords[:, 1].min() - 0.5, coords[:, 1].max() + 0.5
        xx, yy = np.mgrid[x_min_ct:x_max_ct:100j, y_min_ct:y_max_ct:100j]
        xy_test = np.vstack([xx.ravel(), yy.ravel()]).T
        Z = np.exp(kde.score_samples(xy_test)).reshape(xx.shape)
        peak = Z.max()
        if peak > 0:
            ax.contour(xx, yy, Z, levels=[peak * contour_level_frac],
                       colors=contour_color, linestyles=contour_style,
                       linewidths=1.2, alpha=0.8)

        # Draw labels at the adjusted positions, optionally with percentages.
        lx, ly = adjusted[idx]
        pct = ct_counts.get(ct, 0)
        label_text = f"{ct} ({pct:.1f}%)" if show_pct else ct
        ax.text(lx, ly, label_text, fontsize=label_fontsize,
                ha='center', va='center', color='black',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    # Draw coordinate axes with arrowheads.
    if show_arrows:
        x_all, y_all = umap_coords[:, 0], umap_coords[:, 1]
        x_lo, x_hi = x_all.min(), x_all.max()
        y_lo, y_hi = y_all.min(), y_all.max()
        x_range = x_hi - x_lo
        y_range = y_hi - y_lo
        ox = x_lo - x_range * 0.02
        oy = y_lo - y_range * 0.02
        dx = x_range * arrow_frac
        dy = y_range * arrow_frac
        ax.annotate('', xy=(ox + dx, oy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.annotate('', xy=(ox, oy + dy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.text(ox + dx / 2, oy - y_range * 0.03, 'UMAP1',
                fontsize=10, ha='center', va='top')
        ax.text(ox - x_range * 0.03, oy + dy / 2, 'UMAP2',
                fontsize=10, ha='right', va='center', rotation=90)

    # Plot title
    if title is None:
        title = f"Overall (n = {adata.n_obs:,})"
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    return fig, ax

print('Combined enhance_umap function defined')

In [ ]:
# UMAP after annotation revision, combining dashed KDE contours, percentage labels and axes with arrowheads.
fig, ax = enhance_umap(adata, color='cell_type', figsize=(10, 10), size=15,
                       kde_bandwidth=0.3, label_fontsize=9,
                       label_min_dist=2.5, show_pct=True, show_arrows=True)
plt.savefig(FIG_DIR / 'umap_celltype_enhanced.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Histogram of doublet_score with the threshold marked.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) Overall distribution of doublet_score
axes[0].hist(adata.obs['doublet_score'], bins=100, color='steelblue', edgecolor='white')
for thr in [0.15, 0.20, 0.25]:
    n_above = (adata.obs['doublet_score'] > thr).sum()
    axes[0].axvline(thr, color='red', ls='--', lw=1.2)
    axes[0].text(thr + 0.002, axes[0].get_ylim()[1] * 0.9,
                 f'{thr} ({n_above:,})', fontsize=9, color='red')
axes[0].set_xlabel('doublet_score')
axes[0].set_ylabel('cells')
axes[0].set_title('Doublet Score Distribution')

# 2) Box plots of doublet_score by cell_type
import seaborn as sns
order = adata.obs['cell_type'].value_counts().index.tolist()
sns.boxplot(data=adata.obs, x='cell_type', y='doublet_score', ax=axes[1], order=order)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Doublet Score by Cell Type')
plt.tight_layout()
plt.savefig(FIG_DIR / 'doublet_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Inspect the scores for individual cell types in cluster 2.
print("Cluster 2 scores for all cell types (descending):")
scores_c2 = df_v3.loc['2'].sort_values(ascending=False)
for ct, s in scores_c2.items():
    print(f"  {ct:<16s}: {s:.3f}")

# Compare clusters 1 and 2.
print("\nCluster 1 vs 2:")
for ct in ['Macrophage', 'Microglia', 'Monocyte', 'CDC', 'Neutrophil', 'Tumor']:
    print(f"  {ct:<16s}: cl1={df_v3.loc['1', ct]:.3f}  cl2={df_v3.loc['2', ct]:.3f}")

# Inspect key macrophage, microglia and monocyte markers in cluster 2.
key_check = ['CD68', 'LYZ', 'FN1', 'VCAN', 'TMEM119', 'CX3CR1',
             'TREM2', 'GPNMB', 'FCN1', 'S100A8', 'FCER1A', 'HLA-DPB1']
key_valid = [g for g in key_check if g in adata.var_names]

import numpy as np
c2_mask = adata.obs['leiden'] == '2'
c1_mask = adata.obs['leiden'] == '1'
print(f"\nMean expression of key markers (cluster 1 vs 2):")
for g in key_valid:
    idx = adata.var_names.get_loc(g)
    expr_c1 = np.asarray(adata[c1_mask, idx].X.todense()).mean()
    expr_c2 = np.asarray(adata[c2_mask, idx].X.todense()).mean()
    print(f"  {g:<12s}: cl1={expr_c1:.3f}  cl2={expr_c2:.3f}")

## 3. Rank Genes Groups (Wilcoxon)

In [ ]:
import gc

# Exclude ribosomal and mitochondrial genes and broadly expressed lncRNAs.
exclude = (
    adata.var_names.str.startswith(('RPS', 'RPL', 'MT-')) |
    adata.var_names.isin(['MALAT1', 'NEAT1'])
)
print(f'Genes excluded: {exclude.sum()}')

import anndata as ad
adata_clean = ad.AnnData(
    X=adata.X[:, ~exclude],
    obs=adata.obs[['leiden']].copy(),
    var=adata.var[~exclude][[]].copy(),
)
adata_clean.var_names = adata.var_names[~exclude].tolist()
print(f'Genes after filtering: {adata_clean.shape[1]}')

sc.tl.rank_genes_groups(adata_clean, groupby='leiden', method='wilcoxon')
adata.uns['rank_genes_groups'] = adata_clean.uns['rank_genes_groups']
del adata_clean; gc.collect()
print('✅ Wilcoxon rank_genes_groups complete')

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10, save='_rank_genes_panels.pdf')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale='var',
                                dendrogram=False,
                                save='_rank_genes_dotplot.pdf')

In [ ]:
# Tabulate the top five markers per cluster.
top_markers = []
for cl in sorted(adata.obs['leiden'].cat.categories, key=int):
    df = sc.get.rank_genes_groups_df(adata, group=cl).head(5)
    df['cluster'] = cl
    top_markers.append(df)

top_df = pd.concat(top_markers)
top_df[['cluster', 'names', 'logfoldchanges', 'pvals_adj', 'scores']].to_csv(
    FIG_DIR / 'top5_markers_per_cluster.csv', index=False)
print(top_df.pivot_table(index='cluster', columns=top_df.groupby('cluster').cumcount(),
                         values='names', aggfunc='first').to_string())

## 6. Save adata2.h5ad

In [ ]:
# Wilcoxon on cell_type
import gc

exclude = (
    adata.var_names.str.startswith(('RPS', 'RPL', 'MT-')) |
    adata.var_names.isin(['MALAT1', 'NEAT1'])
)
adata_clean = ad.AnnData(
    X=adata.X[:, ~exclude],
    obs=adata.obs[['cell_type']].copy(),
    var=adata.var[~exclude][[]].copy(),
)
adata_clean.var_names = adata.var_names[~exclude].tolist()

sc.tl.rank_genes_groups(adata_clean, groupby='cell_type', method='wilcoxon')
adata.uns['rank_genes_groups_celltype'] = adata_clean.uns['rank_genes_groups']
del adata_clean; gc.collect()
print('✅ Wilcoxon test by cell_type complete')

In [ ]:
# dendrogram
sc.tl.dendrogram(adata, 'cell_type', use_rep='X_pca_harmony', n_pcs=30)

# Save the annotated object.
out_path = WRITE_DIR / 'adata2.h5ad'
adata.write(out_path)
print(f'✅ Saved: {out_path}')
print(f'   {adata.shape[0]} cells × {adata.shape[1]} genes')
print(f'   cell_type: {adata.obs["cell_type"].nunique()} categories')
adata